# INV-M365-N — Approval Contract Alignment

**Purpose:** Establish notebook-backed traceability for the standalone M365 approval path by proving that approval storage and approval Graph authentication resolve through the selected tenant contract instead of a parallel `.env`-only path.

**Lemma mapping:** `L12`

**Invariant support:** tenant-carried approval target configuration, tenant-selected app-only approval authentication, deterministic approval site/list resolution, deterministic approval-item lookup, and bounded `C1A`/`C1C` approval shell inputs.

**Expected results:** The SMARTHAUS approval target is bound to the Operations site, approval auth uses the tenant-selected app-only executor, and approval-item lookup remains deterministic even when SharePoint-backed Graph routes reject an OData field filter for `ApprovalId`.

## Live Failure and Deterministic Fallback

The live `C1C` approval write row proved that SharePoint list-item creation succeeds once the required columns exist, but the follow-up Graph query using `$filter=fields/ApprovalId eq '<uuid>'` can still fail with HTTP `400` on the SharePoint-backed route. The bounded deterministic fallback is therefore to list expanded item fields and scan for the `ApprovalId` locally until the matching item is found.


In [ ]:
def find_item_by_approval_id(pages, approval_id):
    for page in pages:
        for item in page.get("value", []):
            fields = item.get("fields", {})
            if (fields.get("ApprovalId") or item.get("id")) == approval_id:
                return item
    return None


pages = [
    {
        "value": [
            {"id": "11", "fields": {"ApprovalId": "a-1", "Status": "pending"}},
            {"id": "12", "fields": {"ApprovalId": "a-2", "Status": "approved"}},
        ]
    },
    {
        "value": [
            {"id": "13", "fields": {"ApprovalId": "a-3", "Status": "pending"}},
        ]
    },
]

match = find_item_by_approval_id(pages, "a-3")
assert match is not None
assert match["id"] == "13"
assert match["fields"]["Status"] == "pending"
assert find_item_by_approval_id(pages, "missing") is None
